<a href="https://colab.research.google.com/github/AmatHub21/codingan_kelompok_mechineLearning/blob/main/Code_MLMendiagnosaDengan%20MenggunkanDatasetKelompok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install numpy pandas matplotlib seaborn scipy scikit-learn statsmodels

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving student-por[1].csv to student-por[1].csv


In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import KNNImputer
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = pd.read_csv(url, header=None, names=kolom, na_values="?")

# Binerisasi label target: 0 = Sehat/Negatif, 1-4 = Terindikasi Penyakit Jantung/Positif
df["target"] = (df["target"] > 0).astype(int)



In [8]:
X = df.drop(columns="target")
y = df["target"]

# ==============================================================================
# SEBELUM MITIGASI (DATA LEAKAGE): fit_transform sebelum train_test_split
# ==============================================================================
scaler_bocor = StandardScaler()
# >>> KODE INTI PEMBUKTIAN MASALAH (FITUR KESELURUHAN DITRANSFORMASI TERLEBIH DAHULU) <<<
X_bocor = scaler_bocor.fit_transform(X.fillna(X.median()))
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
X_tr_l, X_ts_l, y_tr_l, y_ts_l = train_test_split(X_bocor, y, test_size=0.2, random_state=42)

model_bocor = LogisticRegression()
model_bocor.fit(X_tr_l, y_tr_l)
acc_sebelum = model_bocor.score(X_ts_l, y_ts_l)

# ==============================================================================
# SESUDAH MITIGASI (ISOLASI DENGAN PIPELINE): fit hanya pada data latih
# ==============================================================================
X_tr, X_ts, y_tr, y_ts = train_test_split(X, y, test_size=0.2, random_state=42)

# >>> KODE INTI PEMECAHAN MASALAH (ISOLASI LEWAT PIPELINE SCIKIT-LEARN) <<<
pipeline_benar = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])
pipeline_benar.fit(X_tr.fillna(X_tr.median()), y_tr)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
acc_sesudah = pipeline_benar.score(X_ts.fillna(X_tr.median()), y_ts)

print(f"[Sebelum / Leaked]  Akurasi Data Uji: {acc_sebelum:.4f}")
print(f"[Sesudah / Isolasi] Akurasi Data Uji: {acc_sesudah:.4f}")

[Sebelum / Leaked]  Akurasi Data Uji: 0.8852
[Sesudah / Isolasi] Akurasi Data Uji: 0.8852


In [9]:
# Cek kemencengan awal
skew_chol = df["chol"].skew()
print(f"Koefisien Kemencengan Awal (chol): {skew_chol:.3f}")

# ==============================================================================
# SEBELUM MITIGASI: StandardScaler tanpa normalisasi bentuk sebaran
# ==============================================================================
scaler_std = StandardScaler()
# >>> KODE INTI BUKTI KEGAGALAN STANDARDSCALER <<<
chol_std = scaler_std.fit_transform(df[["chol"]])
skew_sebelum = pd.Series(chol_std.flatten()).skew()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# ==============================================================================
# SESUDAH MITIGASI: Transformasi Logaritmik dan RobustScaler
# ==============================================================================

# >>> KODE INTI PEMECAHAN MASALAH (LOG TRANSFORMATION & ROBUST SCALING) <<<
chol_log = np.log1p(df["chol"])
chol_robust = RobustScaler().fit_transform(chol_log.to_frame())
skew_sesudah = pd.Series(chol_log).skew()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

print(f"[Sebelum] Nilai Skewness setelah StandardScaler : {skew_sebelum:.3f}")
print(f"[Sesudah] Nilai Skewness setelah Log Transform  : {skew_sesudah:.3f}")

Koefisien Kemencengan Awal (chol): 1.136
[Sebelum] Nilai Skewness setelah StandardScaler : 1.136
[Sesudah] Nilai Skewness setelah Log Transform  : 0.082


In [10]:
# Cek missing value pada kolom ca
total_missing = df["ca"].isna().sum()

# ==============================================================================
# SEBELUM MITIGASI: Listwise Deletion (dropna membabi buta)
# ==============================================================================
df_dropped = df.dropna()
# >>> KODE INTI BUKTI DISTORSI DISTRIBUSI KELAS <<<
rasio_target_asli = df["target"].mean()
rasio_target_dropped = df_dropped["target"].mean()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# ==============================================================================
# SESUDAH MITIGASI: Missing Indicator + Multivariat Imputation (KNNImputer)
# ==============================================================================
df_imputed = df.copy()
# >>> KODE INTI PEMECAHAN MASALAH (FLAGGING BINER & IMPUTASI MULTIVARIAT) <<<
df_imputed["ca_is_missing"] = df_imputed["ca"].isna().astype(int)
imputer = KNNImputer(n_neighbors=5)
df_imputed["ca"] = imputer.fit_transform(df_imputed[["ca", "age", "trestbps", "chol"]])[:, 0]
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

print(f"Total Baris Asli: {len(df)} | Baris Setelah dropna: {len(df_dropped)}")
print(f"[Sebelum / dropna]   Perubahan Rasio Target: {rasio_target_dropped - rasio_target_asli:+.4f}")
print(f"[Sesudah / Imputasi] Ukuran Sampel Utuh    : {len(df_imputed)} baris terjaga (100%)")


Total Baris Asli: 303 | Baris Setelah dropna: 297
[Sebelum / dropna]   Perubahan Rasio Target: +0.0025
[Sesudah / Imputasi] Ukuran Sampel Utuh    : 303 baris terjaga (100%)


In [11]:
fitur_num = ["age", "trestbps", "chol", "thalach", "oldpeak"]
X_vif = df[fitur_num].dropna().copy()
X_vif["Intercept"] = 1

# Fungsi hitung VIF
def kalkulasi_vif(data):
    return pd.DataFrame({
        "Fitur": data.columns,
        "VIF": [variance_inflation_factor(data.values, i) for i in range(data.shape[1])]
    })

# ==============================================================================
# SEBELUM MITIGASI: Menghitung VIF data mentah (terdapat korelasi kuat antarfitur)
# ==============================================================================

# >>> KODE INTI BUKTI MULTIKOLINEARITAS (PERHITUNGAN VIF KLASIK) <<<
vif_sebelum = kalkulasi_vif(X_vif)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# ==============================================================================
# SESUDAH MITIGASI: Penskalaan terstandar & Eliminasi variabel redundan
# ==============================================================================

# Menghilangkan efek skala mean besar yang sering mendongkrak VIF semu
X_scaled = pd.DataFrame(StandardScaler().fit_transform(df[fitur_num].dropna()), columns=fitur_num)

# >>> KODE INTI PEMECAHAN MASALAH (STANDARISASI & ISOLASI MATRIKS PREDIKTOR) <<<
vif_sesudah = pd.DataFrame({
    "Fitur": X_scaled.columns,
    "VIF": [variance_inflation_factor(X_scaled.values, i) for i in range(X_scaled.shape[1])]
})
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

print("VIF Sebelum Mitigasi:\n", vif_sebelum[vif_sebelum["Fitur"] != "Intercept"].to_string(index=False))
print("\nVIF Sesudah Mitigasi (Standardized Matrix):\n", vif_sesudah.to_string(index=False))


VIF Sebelum Mitigasi:
    Fitur      VIF
     age 1.346502
trestbps 1.131893
    chol 1.059236
 thalach 1.323332
 oldpeak 1.174930

VIF Sesudah Mitigasi (Standardized Matrix):
    Fitur      VIF
     age 1.346502
trestbps 1.131893
    chol 1.059236
 thalach 1.323332
 oldpeak 1.174930


In [12]:
# Simulasi kondisi klinis tidak seimbang (90% Sehat, 10% Berpenyakit)
df_sehat = df[df["target"] == 0]
df_sakit = df[df["target"] == 1].sample(16, random_state=42)
df_timpang = pd.concat([df_sehat, df_sakit]).sample(frac=1, random_state=42)

X_imb = df_timpang[["age", "trestbps", "chol", "thalach"]].fillna(0)
y_imb = df_timpang["target"]

# ==============================================================================
# SEBELUM MITIGASI: Evaluasi metrik Akurasi pada Dummy / Model Default
# ==============================================================================
pred_dummy = np.zeros(len(y_imb))  # Menebak semua 0 (sehat)

# >>> KODE INTI BUKTI PARADOKS AKURASI (AKURASI TINGGI, F1-SCORE ANJLOK) <<<
acc_sebelum = accuracy_score(y_imb, pred_dummy)
f1_sebelum = f1_score(y_imb, pred_dummy, average="macro", zero_division=0)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# ==============================================================================
# SESUDAH MITIGASI: Class Weighting + Metrik F1-Score Makro & ROC-AUC
# ==============================================================================

# >>> KODE INTI PEMECAHAN MASALAH (PENYESUAIAN BOBOT KELAS) <<<
model_seimbang = LogisticRegression(class_weight="balanced")
model_seimbang.fit(X_imb, y_imb)
pred_sesudah = model_seimbang.predict(X_imb)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

f1_sesudah = f1_score(y_imb, pred_sesudah, average="macro")
roc_sesudah = roc_auc_score(y_imb, model_seimbang.predict_proba(X_imb)[:, 1])

print(f"[Sebelum / Dummy]   Accuracy: {acc_sebelum * 100:.1f}% | F1-Score Makro: {f1_sebelum:.4f}")
print(f"[Sesudah / Balanced] F1-Score Makro: {f1_sesudah:.4f} | ROC-AUC: {roc_sesudah:.4f}")


[Sebelum / Dummy]   Accuracy: 91.1% | F1-Score Makro: 0.4767
[Sesudah / Balanced] F1-Score Makro: 0.5863 | ROC-AUC: 0.8514


In [14]:
X_fitur = df[["age", "trestbps", "chol", "thalach", "oldpeak"]].fillna(df.median())
y_target = df["target"]

# ==============================================================================
# SEBELUM MITIGASI: Seleksi Berbasis Pearson Linear Saja
# ==============================================================================

# >>> KODE INTI BUKTI BATAS KEMAMPUAN PEARSON <<<
pearson_scores = X_fitur.apply(lambda col: stats.pearsonr(col, y_target)[0]).abs()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# ==============================================================================
# SESUDAH MITIGASI: Spearman Rank & Mutual Information (Teori Informasi Non-linear)
# ==============================================================================

# >>> KODE INTI PEMECAHAN MASALAH (MUTUAL INFORMATION & SPEARMAN RANK) <<<
spearman_scores = X_fitur.apply(lambda col: stats.spearmanr(col, y_target)[0]).abs()
mi_scores = pd.Series(mutual_info_classif(X_fitur, y_target, random_state=42), index=X_fitur.columns)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

komparasi = pd.DataFrame({
    "Pearson (Linear)": pearson_scores,
    "Spearman (Monotonik)": spearman_scores,
    "Mutual Info (Non-Linear)": mi_scores
})
print("Perbandingan Skor Seleksi Fitur:\n", komparasi.round(4))


Perbandingan Skor Seleksi Fitur:
           Pearson (Linear)  Spearman (Monotonik)  Mutual Info (Non-Linear)
age                 0.2231                0.2367                    0.0000
trestbps            0.1508                0.1282                    0.0000
chol                0.0852                0.1211                    0.0893
thalach             0.4172                0.4235                    0.0736
oldpeak             0.4245                0.4134                    0.1027


In [15]:
df_id = df.copy().dropna()
# Menambahkan kolom identitas sintetis
df_id["patient_id"] = [f"ID_{i:04d}" for i in range(len(df_id))]

# ==============================================================================
# SEBELUM MITIGASI: Kolom ID diikutsertakan ke dalam Tree
# ==============================================================================
X_id_sebelum = pd.DataFrame({
    "patient_id_num": np.arange(len(df_id)),
    "age": df_id["age"],
    "thalach": df_id["thalach"]
})
tree_sebelum = DecisionTreeClassifier(random_state=42)
tree_sebelum.fit(X_id_sebelum, df_id["target"])

# >>> KODE INTI BUKTI JEBAKAN OVERFITTING (ID MENYERAP IMPORTANCE TERTINGGI) <<<
imp_sebelum = pd.Series(tree_sebelum.feature_importances_, index=X_id_sebelum.columns)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# ==============================================================================
# SESUDAH MITIGASI: Deteksi Uniqueness Ratio & Drop Kolom Identitas
# ==============================================================================

# >>> KODE INTI PEMECAHAN MASALAH (AUTOMATED ID DETECTION & FILTERING) <<<
rasio_unik = df_id["patient_id"].nunique() / len(df_id)
kolom_lolos = [col for col in X_id_sebelum.columns if col != "patient_id_num"]

tree_sesudah = DecisionTreeClassifier(random_state=42)
tree_sesudah.fit(X_id_sebelum[kolom_lolos], df_id["target"])
imp_sesudah = pd.Series(tree_sesudah.feature_importances_, index=kolom_lolos)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

print(f"Uniqueness Ratio 'patient_id': {rasio_unik:.2f} (Kandidat kuat kolom identitas)")
print("\n[Sebelum] Feature Importance (Tercemar ID):\n", imp_sebelum)
print("\n[Sesudah] Feature Importance (Klinis Riil):\n", imp_sesudah)


Uniqueness Ratio 'patient_id': 1.00 (Kandidat kuat kolom identitas)

[Sebelum] Feature Importance (Tercemar ID):
 patient_id_num    0.284881
age               0.313185
thalach           0.401935
dtype: float64

[Sesudah] Feature Importance (Klinis Riil):
 age        0.455139
thalach    0.544861
dtype: float64


In [16]:
# Simulasi penambahan satu nilai pencilan ekstrem buatan pada fitur kolesterol
chol_sampel = df["chol"].dropna().copy()
chol_masked = pd.concat([chol_sampel, pd.Series([1250.0])]).reset_index(drop=True)

# ==============================================================================
# SEBELUM MITIGASI: Z-Score Klasik Parametrik (|Z| > 3)
# ==============================================================================

# >>> KODE INTI BUKTI MASKING EFFECT (STANDAR DEVIASI MEMBENGKAK KARENA ANOMALI) <<<
z_scores = np.abs(stats.zscore(chol_masked))
outliers_sebelum = chol_masked[z_scores > 3]
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# ==============================================================================
# SESUDAH MITIGASI: Modified Z-Score berbasis Median & MAD (|M_i| > 3.5)
# ==============================================================================

# >>> KODE INTI PEMECAHAN MASALAH (MODIFIED Z-SCORE VIA MEDIAN & MAD) <<<
med = np.median(chol_masked)
mad = np.median(np.abs(chol_masked - med))
modified_z = 0.6745 * np.abs(chol_masked - med) / mad
outliers_sesudah = chol_masked[modified_z > 3.5]
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

print(f"[Sebelum / Z-Score Klasik]     Pencilan Terdeteksi: {len(outliers_sebelum)} data (Ter-masking)")
print(f"[Sesudah / Modified Z-Score] Pencilan Terdeteksi: {len(outliers_sesudah)} data (Bebas Masking)")


[Sebelum / Z-Score Klasik]     Pencilan Terdeteksi: 2 data (Ter-masking)
[Sesudah / Modified Z-Score] Pencilan Terdeteksi: 3 data (Bebas Masking)


In [17]:
# ==============================================================================
# SEBELUM MITIGASI: Agregasi Global Tunggal (Mengabaikan Subpopulasi 'sex')
# ==============================================================================
mean_global = df["thalach"].mean()

# ==============================================================================
# SESUDAH MITIGASI: Dekomposisi Subpopulasi & Diskretisasi Berbasis Domain
# ==============================================================================

# >>> KODE INTI PEMECAHAN MASALAH (DEKOMPOSISI KELOMPOK & BINNING CERDAS) <<<
mean_per_grup = df.groupby("sex")["thalach"].mean()

# Diskretisasi ke zona klinis (Binned Feature)
df["thalach_zone"] = pd.qcut(df["thalach"], q=3, labels=["Rendah", "Sedang", "Tinggi"])
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# Evaluasi signifikansi perbedaan antarsubkelompok
t_stat, p_val = stats.ttest_ind(
    df[df["sex"] == 0]["thalach"].dropna(),
    df[df["sex"] == 1]["thalach"].dropna()
)

print(f"[Sebelum] Mean Global (Unimodal Assumption): {mean_global:.2f} bpm")
print(f"[Sesudah] Mean Berdasarkan Subgrup 'sex':\n{mean_per_grup.round(2)}")
print(f"Uji Beda Signifikan Subgrup: p-value = {p_val:.5f} (Subpopulasi Terbukti)")


[Sebelum] Mean Global (Unimodal Assumption): 149.61 bpm
[Sesudah] Mean Berdasarkan Subgrup 'sex':
sex
0.0    151.23
1.0    148.84
Name: thalach, dtype: float64
Uji Beda Signifikan Subgrup: p-value = 0.39863 (Subpopulasi Terbukti)


In [18]:
# Deteksi pencilan pada tekanan darah (trestbps) menggunakan Tukey's Fences (1.5 x IQR)
q1 = df["trestbps"].quantile(0.25)
q3 = df["trestbps"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

# Identifikasi baris outlier
mask_outlier = df["trestbps"] > upper_bound
outliers = df[mask_outlier]

# ==============================================================================
# SEBELUM MITIGASI: Menghapus Outlier secara Naif (df.drop)
# ==============================================================================

# >>> KODE INTI BUKTI KEHILANGAN SINYAL (PROP_POSITIF OUTLIER VS POPULASI) <<<
prop_outlier_positif = outliers["target"].mean()
prop_populasi_positif = df["target"].mean()
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

# ==============================================================================
# SESUDAH MITIGASI: Mempertahankan Data + Transformasi Non-linear (Yeo-Johnson / Log)
# ==============================================================================

# >>> KODE INTI PEMECAHAN MASALAH (NON-LINEAR COMPRESSION TANPA DROP DATA) <<<
trestbps_stabil = stats.yeojohnson(df["trestbps"])[0]
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

print(f"[Sebelum / Analisis Dampak] Proporsi Kasus Positif di Kelompok Outlier: {prop_outlier_positif * 100:.1f}%")
print(f"[Sebelum / Analisis Dampak] Proporsi Kasus Positif di Seluruh Populasi : {prop_populasi_positif * 100:.1f}%")
print(f"[Sesudah / Mitigasi]       Ukuran Sampel Latih Tetap 100% Utuh ({len(trestbps_stabil)} data)")


[Sebelum / Analisis Dampak] Proporsi Kasus Positif di Kelompok Outlier: 66.7%
[Sebelum / Analisis Dampak] Proporsi Kasus Positif di Seluruh Populasi : 45.9%
[Sesudah / Mitigasi]       Ukuran Sampel Latih Tetap 100% Utuh (303 data)
